# NER 项目上手演示（单文档、零文件写入）

这个 Notebook 面向第一次接触该工程的同学：
- 跑通一遍完整链路（读取 -> Prompt -> 推理 -> 解析纠偏 -> BIO -> 复核视图）
- 只处理 1 个文档中的 1 条文本
- 全过程只在内存中演示，不写入任何新文件，避免污染工程目录

## Step 1: 环境与路径初始化
这一步只做路径配置和基础导入。

In [1]:
from pathlib import Path
import json
import sys

PROJECT_ROOT = Path('/home/superuser/dev/NER/ner_dataset_builder')
DATA_DIR = Path('/home/superuser/dev/NER/data')
PROMPT_CONFIG = PROJECT_ROOT / 'configs/prompt_config.json'
FEW_SHOTS = PROJECT_ROOT / 'configs/few_shots.yaml'
RULES_CONFIG = PROJECT_ROOT / 'configs/entity_rules.yaml'
MODEL_PATH = '/home/superuser/LLM_Model/Qwen3.5-4B'

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('PROJECT_ROOT =', PROJECT_ROOT)
print('DATA_DIR =', DATA_DIR)
print('PROMPT_CONFIG exists =', PROMPT_CONFIG.exists())
print('FEW_SHOTS exists =', FEW_SHOTS.exists())
print('RULES_CONFIG exists =', RULES_CONFIG.exists())

PROJECT_ROOT = /home/superuser/dev/NER/ner_dataset_builder
DATA_DIR = /home/superuser/dev/NER/data
PROMPT_CONFIG exists = True
FEW_SHOTS exists = True
RULES_CONFIG exists = True


## Step 2: 查看 Prompt 相关配置输入
`prompt_config.json` 和 `few_shots.yaml` 是模型提问方式的核心。

In [2]:
prompt_cfg = json.loads(PROMPT_CONFIG.read_text(encoding='utf-8'))
print('system_instruction:')
print(prompt_cfg['system_instruction'])
print('\n---\ntask_description:')
print(prompt_cfg['task_description'])

system_instruction:
你是一个严谨的地质与石油领域数据标注专家。你的任务是从文本中抽取特定的实体，并输出带有严格格式的JSON。

---
task_description:
请抽取以下六类实体：
1. well_name (井名)
2. block (区块)
3. strat_unit (层位名称)
4. lithology (主岩性)
5. lithology_color (岩性颜色)
6. lithology_texture (岩性结构描述)

严格要求：
- 必须输出纯JSON格式，不要包含任何额外的解释或Markdown代码块。
- 输出格式为：{"实体类别": {"实体原文": [[起始索引, 结束索引]]}}。
- 如果未找到任何实体，请输出 {}。


## Step 3: 读取单文档并取 1 条样本
这里会调用工程的 DataLoader，注意它已包含 `<table>...</table>` 过滤逻辑。

In [3]:
from core import DataLoader

loader = DataLoader(str(DATA_DIR))
files = sorted(DATA_DIR.glob('*.md'))
assert files, '未找到输入数据文件'

one_file = files[0]
items = loader.load_items_from_file(one_file)
assert items, f'{one_file.name} 中没有可用样本'

sample = items[0]
text = sample['text']

print('source_file =', sample['source_file'])
print('page_number =', sample['page_number'])
print('block_index =', sample['block_index'])
print('text_length =', len(text))
print('\ntext_preview:\n', text[:300])

source_file = SN0006-01 井录井报告.md
page_number = 1
block_index = 0
text_length = 51

text_preview:
 # <br> 鄂尔多斯盆地伊陕斜坡 <br>

# <br> SN0006-01 井录井报告 <br>


## Step 4: 构建最终 Prompt（内存中）
这一步验证 `system_instruction + task_description + few_shots + input_text` 的拼接效果。

In [4]:
from core import PromptBuilder

pb = PromptBuilder(str(PROMPT_CONFIG), str(FEW_SHOTS))
prompt = pb.build_prompt(text)

print('prompt_length =', len(prompt))
print('\nprompt_preview:\n')
print(prompt[:1200])

prompt_length = 887

prompt_preview:

你是一个严谨的地质与石油领域数据标注专家。你的任务是从文本中抽取特定的实体，并输出带有严格格式的JSON。

请抽取以下六类实体：
1. well_name (井名)
2. block (区块)
3. strat_unit (层位名称)
4. lithology (主岩性)
5. lithology_color (岩性颜色)
6. lithology_texture (岩性结构描述)

严格要求：
- 必须输出纯JSON格式，不要包含任何额外的解释或Markdown代码块。
- 输出格式为：{"实体类别": {"实体原文": [[起始索引, 结束索引]]}}。
- 如果未找到任何实体，请输出 {}。

### 样例参考 ###
text: "SN0015-08 井录井完井报告"
label: {"well_name": {"SN0015-08": [[0, 8]]}}

text: "评审单位：长庆油田苏里格南作业分公司地学部"
label: {"block": {"苏里格南作业分公司": [[9, 17]]}}

text: "古生界二叠系下统山西组"
label: {"strat_unit": {"山西组": [[8, 10]]}}

text: "云岩：成份中白云石约占85.2~90.4%"
label: {"lithology": {"云岩": [[0, 1]]}}

text: "岩性特征：棕红色泥岩、砂质泥岩，浅棕红色细砂岩、泥质砂岩"
label: {"lithology_color": {"棕红色": [[5, 7]], "浅棕红色": [[16, 19]]}}

text: "岩性特征：棕红色泥岩、砂质泥岩，浅棕红色细砂岩、泥质砂岩"
label: {"lithology_texture": {"砂质": [[11, 12]], "泥质": [[24, 25]]}}

### 当前任务 ###
text: "# <br> 鄂尔多斯盆地伊陕斜坡 <br>

# <br> SN0006-01 井录井报告 <br>"
label: 


## Step 5: 单条推理（演示版）
加载模型并对这 1 条文本推理。为了演示速度，这里覆盖 `max_new_tokens`。

In [5]:
from core import QwenModelEngine

engine = QwenModelEngine(MODEL_PATH, str(PROMPT_CONFIG))
model_output = engine.generate_text(prompt, max_new_tokens=128)

print('model_output_preview:\n')
print(model_output[:1200])

Loading weights:   0%|          | 0/426 [00:00<?, ?it/s]

model_output_preview:

<think>

</think>

```json
{
    "well_name": {
        "SN0006-01": [
            20,
            28
        ]
    },
    "block": {
        "鄂尔多斯盆地伊陕斜坡": [
            3,
            24
        ]
    }
}
```


## Step 6: 解析 + Offset 纠偏 + 规则过滤
这一步会忽略模型给的索引，按原文重算偏移；并输出规则过滤审计信息。

In [6]:
from core import LLMOutputParser

parser = LLMOutputParser(str(RULES_CONFIG))
corrected_entities, filtered_entities = parser.parse_and_correct_with_audit(text, model_output)

print('corrected_entities =')
print(json.dumps(corrected_entities, ensure_ascii=False, indent=2))
print('\nfiltered_entities =')
print(json.dumps(filtered_entities, ensure_ascii=False, indent=2))

corrected_entities =
{
  "well_name": {
    "SN0006-01": [
      [
        31,
        39
      ]
    ]
  },
  "block": {
    "鄂尔多斯盆地伊陕斜坡": [
      [
        7,
        16
      ]
    ]
  }
}

filtered_entities =
[]


## Step 7: 转成 BIO（内存中，不落盘）
这里生成与 `train.bio` 同格式的单样本 BIO 行。

In [7]:
from core import BIOConverter

bio_lines = BIOConverter.to_bio_lines(text, corrected_entities)

print('bio_lines_count =', len(bio_lines))
print('\nBIO preview:')
for line in bio_lines[:80]:
    print(line)

bio_lines_count = 52

BIO preview:
# O
  O
< O
b O
r O
> O
  O
鄂 B-block
尔 I-block
多 I-block
斯 I-block
盆 I-block
地 I-block
伊 I-block
陕 I-block
斜 I-block
坡 I-block
  O
< O
b O
r O
> O

 O

 O
# O
  O
< O
b O
r O
> O
  O
S B-well_name
N I-well_name
0 I-well_name
0 I-well_name
0 I-well_name
6 I-well_name
- I-well_name
0 I-well_name
1 I-well_name
  O
井 O
录 O
井 O
报 O
告 O
  O
< O
b O
r O
> O



## Step 8: 构造复核视图（内存中）
模拟 `bi_only` 与 `bi_spans` 的核心结果，让新手理解最终 I/O 长什么样。

In [8]:
from extract_bi_for_review import parse_bio_tokens

raw_bio_text = '\n'.join(bio_lines)
tokens = parse_bio_tokens(raw_bio_text)

bi_only_rows = []
spans = []
cur_type = ''
cur_start = -1
cur_chars = []

def flush(end_idx):
    global_dummy = None
    if cur_type and cur_start >= 0 and cur_chars:
        spans.append({
            'entity_type': cur_type,
            'start_idx': cur_start,
            'end_idx': end_idx,
            'text': ''.join(cur_chars)
        })

for i, (ch, tag) in enumerate(tokens):
    if tag.startswith(('B-', 'I-')):
        bi_only_rows.append({'token_idx': i, 'char': ch, 'tag': tag})
    if tag.startswith('B-'):
        flush(i - 1)
        cur_type = tag[2:]
        cur_start = i
        cur_chars = [ch]
    elif tag.startswith('I-'):
        t = tag[2:]
        if cur_type == t and cur_start >= 0:
            cur_chars.append(ch)
        else:
            flush(i - 1)
            cur_type = t
            cur_start = i
            cur_chars = [ch]
    else:
        flush(i - 1)
        cur_type = ''
        cur_start = -1
        cur_chars = []

flush(len(tokens) - 1)

print('bi_only_rows (preview):')
for row in bi_only_rows[:20]:
    print(row)

print('\nspans (preview):')
for row in spans[:20]:
    print(row)

bi_only_rows (preview):
{'token_idx': 7, 'char': '鄂', 'tag': 'B-block'}
{'token_idx': 8, 'char': '尔', 'tag': 'I-block'}
{'token_idx': 9, 'char': '多', 'tag': 'I-block'}
{'token_idx': 10, 'char': '斯', 'tag': 'I-block'}
{'token_idx': 11, 'char': '盆', 'tag': 'I-block'}
{'token_idx': 12, 'char': '地', 'tag': 'I-block'}
{'token_idx': 13, 'char': '伊', 'tag': 'I-block'}
{'token_idx': 14, 'char': '陕', 'tag': 'I-block'}
{'token_idx': 15, 'char': '斜', 'tag': 'I-block'}
{'token_idx': 16, 'char': '坡', 'tag': 'I-block'}
{'token_idx': 31, 'char': 'S', 'tag': 'B-well_name'}
{'token_idx': 32, 'char': 'N', 'tag': 'I-well_name'}
{'token_idx': 33, 'char': '0', 'tag': 'I-well_name'}
{'token_idx': 34, 'char': '0', 'tag': 'I-well_name'}
{'token_idx': 35, 'char': '0', 'tag': 'I-well_name'}
{'token_idx': 36, 'char': '6', 'tag': 'I-well_name'}
{'token_idx': 37, 'char': '-', 'tag': 'I-well_name'}
{'token_idx': 38, 'char': '0', 'tag': 'I-well_name'}
{'token_idx': 39, 'char': '1', 'tag': 'I-well_name'}

spans (prev

## Step 9: 总结（新手应建立的 I/O 认知）
- 输入最小单元：`page_text`（已过滤 `<table>` 区块）。
- Prompt 由 `prompt_config + few_shots + input_text` 拼装。
- 模型输出 JSON 后，必须经过“解析纠偏 + 规则过滤”。
- 训练核心格式是 BIO；复核核心视图是 `bi_only` 与 `spans`。
- 本 Notebook 全过程不写文件，正式产出请按 `MANUAL_GEO.md` 执行全流程命令。